In [41]:
SELECT count(*) as number_of_tables
FROM information_schema.tables
WHERE table_type = 'BASE TABLE'

How many records do we actually have in our main transactions tables?? 

In [42]:
select count(*) as number_of_rows
from factinternetsales;
GO
select count(*) as number_of_rows
from factresellersales;

In [43]:
drop view if exists Orders_by_category;
GO
CREATE VIEW Orders_by_category AS
    select englishproductcategoryname,englishproductsubcategoryname,sum(OrderQuantity) as total_order_quantity
    from factinternetsales
    LEFT JOIN dimproduct
    ON factinternetsales.productkey = dimproduct.ProductKey
    left join dimproductsubcategory
    on dimproduct.ProductSubcategoryKey = dimproductsubcategory.ProductSubcategoryKey
    left join dimproductcategory
    on dimproductsubcategory.productcategorykey = dimproductcategory.productcategorykey
    --where dimproduct.ProductSubcategoryKey is not null 
    and dimproductcategory.englishproductcategoryname is not null
    group by englishproductsubcategoryname,englishproductcategoryname;
    --group by englishproductcategoryname;
go
select * from Orders_by_category;

In [50]:
select min(orderdate) as first_order_date,max(orderdate) as last_order_date
from factresellersales
UNION ALL
select min(orderdate) as first_order_date,max(orderdate) as last_order_date
from factinternetsales
;

In [46]:
WITH sales_per_year AS (
    SELECT 
        YEAR(orderdate) AS order_year, 
        SUM(orderquantity) AS total_orders
    FROM factinternetsales
    GROUP BY YEAR(orderdate)
)
SELECT 
    total_orders,
    order_year,
   ROUND(
    (
        total_orders - LAG(total_orders) OVER (ORDER BY order_year ASC)
    )
    / CAST(LAG(total_orders) OVER (ORDER BY order_year ASC) AS DECIMAL(10,2))
    * 100 ,0) AS percentage_change
FROM sales_per_year;

GO

WITH b2b_sales_per_year AS (
    SELECT 
        YEAR(orderdate) AS order_year, 
        SUM(orderquantity) AS total_orders
    FROM factresellersales
    GROUP BY YEAR(orderdate)
)
SELECT 
    total_orders,
    order_year,
   ROUND(
    (
        total_orders - LAG(total_orders) OVER (ORDER BY order_year ASC)
    )
    / CAST(LAG(total_orders) OVER (ORDER BY order_year ASC) AS DECIMAL(10,2))
    * 100 ,0) AS percentage_change
FROM b2b_sales_per_year;

In [53]:
select column_name
from information_schema.columns
where table_name = 'dimproduct';

(36 rows affected)

column_name          
---------------------
ProductKey           
ProductAlternateKey  
ProductSubcategoryKey
WeightUnitMeasureCode
SizeUnitMeasureCode  
EnglishProductName   
SpanishProductName   
FrenchProductName    
StandardCost         
FinishedGoodsFlag    
Color                
SafetyStockLevel     
ReorderPoint         
ListPrice            
Size                 
SizeRange            
Weight               
DaysToManufacture    
ProductLine          
DealerPrice          
Class                
Style                
ModelName            
LargePhoto           
EnglishDescription   
FrenchDescription    
ChineseDescription   
ArabicDescription    
HebrewDescription    
ThaiDescription      
GermanDescription    
JapaneseDescription  
TurkishDescription   
StartDate            
EndDate              
Status               
(36 rows)

Total execution time: 00:00:00.278

In [ ]:
with top_10_orders as 
        (select top 10 EnglishProductName, sum(OrderQuantity) as TotalOrders
        FROM factinternetsales as fs
        LEFT JOIN dimproduct as dp on fs.ProductKey = dp.ProductKey
        group by  EnglishProductName
        order by sum(OrderQuantity) Desc)
    
        select *
        from top_10_orders;
        

with top_10_revenue as (SELECT TOP 10 EnglishProductName,sum(ExtendedAmount) as revenue
        FROM factinternetsales as fs
        LEFT JOIN dimproduct as dp on fs.ProductKey = dp.ProductKey
        group by  EnglishProductName
        order by sum(ExtendedAmount)Desc)

select *
from top_10_revenue;


with top_10_profit as (SELECT TOP 10 EnglishProductName,sum(ExtendedAmount - TotalProductCost) as Profit
        FROM factinternetsales as fs
        LEFT JOIN dimproduct as dp on fs.ProductKey = dp.ProductKey
        group by  EnglishProductName
        order by sum(ExtendedAmount - TotalProductCost) Desc)
        
select top 10 *
from dimproduct as dp
left JOIN top_10_orders on dp.EnglishProductName = top_10_orders.EnglishProductName
left JOIN top_10_profit on dp.EnglishProductName = top_10_profit.EnglishProductName
left JOIN top_10_revenue on dp.EnglishProductName = top_10_revenue.EnglishProductName
;


(10 rows affected)
(10 rows affected)
(10 rows affected)

EnglishProductName      | TotalOrders
------------------------+------------
Water Bottle - 30 oz.   | 4244       
Patch Kit/8 Patches     | 3191       
Mountain Tire Tube      | 3095       
Road Tire Tube          | 2376       
Sport-100 Helmet, Red   | 2230       
AWC Logo Cap            | 2190       
Sport-100 Helmet, Blue  | 2125       
Fender Set - Mountain   | 2121       
Sport-100 Helmet, Black | 2085       
Mountain Bottle Cage    | 2025       
(10 rows)

EnglishProductName      | revenue     
------------------------+-------------
Mountain-200 Black, 46  | 1373469.5482
Mountain-200 Black, 42  | 1363142.0934
Mountain-200 Silver, 38 | 1339462.7904
Mountain-200 Silver, 46 | 1301100.0984
Mountain-200 Black, 38  | 1294866.1412
Mountain-200 Silver, 42 | 1257434.5728
Road-150 Red, 48        | 1205876.99  
Road-150 Red, 62        | 1202298.72  
Road-150 Red, 52        | 1080637.54  
Road-150 Red, 56        | 1055589.65  
(10 row

(10 rows affected)

EnglishProductName      | (No column name)
------------------------+-----------------
Water Bottle - 30 oz.   | 4244            
Patch Kit/8 Patches     | 3191            
Mountain Tire Tube      | 3095            
Road Tire Tube          | 2376            
Sport-100 Helmet, Red   | 2230            
AWC Logo Cap            | 2190            
Sport-100 Helmet, Blue  | 2125            
Fender Set - Mountain   | 2121            
Sport-100 Helmet, Black | 2085            
Mountain Bottle Cage    | 2025            
(10 rows)

Total execution time: 00:00:00.185